# Mobility-Informed Renewal Equations — Parameter Exploration Notebook

**Reference:** Mills, C. (2026). *Reproduction numbers for epidemics on human mobility networks.*
University of Oxford.

This notebook takes a **β-first** approach to parametrising epidemic simulations.
Rather than assuming a known R₀, we start from quantities that are directly observable
or estimable early in an outbreak, and treat R₀ as a *derived output*.

## What you know at the start of an outbreak

| Parameter | Source | Uncertainty |
|-----------|--------|-------------|
| **λ_W** (home contact rate) | POLYMOD, contact surveys | Low (well-measured) |
| **λ_B/λ_W** ratio | Modelling assumption | Medium |
| **p(a_E)** (infectiousness profile) | Literature for similar pathogens | Medium |
| **f^{jk}(t)** (mobility) | Mobile phone data, census | Medium–high |
| **β** (per-contact transmission prob.) | Back-calculated from growth rate | **High** |
| **R₀** | *Derived from above* | — |

## Notebook contents

1. [Setting up realistic parameters](#sec1)
2. [β → R₀ mapping and growth rate inversion](#sec2)
3. [Simulation in β-mode](#sec3)
4. [Univariate sensitivity: sweeping β, λ_B/λ_W, GT, commuting](#sec4)
5. [Bivariate parameter surfaces](#sec5)
6. [Cross-network comparison (urban vs rural)](#sec6)
7. [Uncertainty propagation: β uncertainty → R₀ uncertainty](#sec7)


## 1. Setting up realistic parameters <a id="sec1"></a>

We model a SARS-CoV-2-like directly transmitted pathogen.
Contact rates come from POLYMOD (Mossong 2008); the infectiousness profile
from Hart et al. (2022) *Lancet Infect Dis*.


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import BoundaryNorm
from scipy.stats import gamma as gamma_dist

from mobility_rt_framework import (
    MobilityRtFramework,
    discretise_gamma,
    build_synthetic_network,
    compute_R0_from_beta,
    estimate_beta_from_growth_rate,
    scan_beta_to_R0,
    parameter_sweep,
    run_all_tests,
    spectral_analysis,
    source_sink_analysis,
    OKABE_ITO,
)

# Quick sanity-check
assert run_all_tests(verbose=False), "Framework tests failed"
print("Framework loaded and validated ✓")
OK = OKABE_ITO

In [ ]:
# ── Parameters that are known (or estimable) at outbreak start ──────────────

# Contact rates — from POLYMOD Mossong 2008 PLOS Med (European average)
LAMBDA_W = 13.0          # home-location contacts / person / day
LAMBDA_B_RATIO = 0.30    # lambda_B / lambda_W (modelling assumption: 30%)
LAMBDA_B = LAMBDA_W * LAMBDA_B_RATIO

# Infectiousness profile p(a_E) — Gamma(mean, SD)
#   Hart WS et al. 2022 Lancet Infect Dis — Alpha SARS-CoV-2
GT_MEAN   = 5.5  # days
GT_SD     = 1.8  # days
MAX_DAYS  = 25

p = discretise_gamma(GT_MEAN, GT_SD, MAX_DAYS)
mean_gt   = float((np.arange(MAX_DAYS) * p).sum())
print(f"Mean generation time: {mean_gt:.2f} days")
print(f"λ_W = {LAMBDA_W}, λ_B = {LAMBDA_B:.1f} (ratio = {LAMBDA_B_RATIO})")
print(f"Profile sums to: {p.sum():.8f}")

# Visualise the profile
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(np.arange(MAX_DAYS), p, color=OK[0], edgecolor="none", width=0.85)
ax.axvline(mean_gt, color=OK[5], lw=1.4, ls="--", label=f"Mean = {mean_gt:.1f}d")
ax.set_xlabel("Infection age $a_E$ (days)"); ax.set_ylabel("$p(a_E)$")
ax.set_title("Infectiousness profile — Gamma(5.5d, 1.8d)\n(Hart et al. 2022 Lancet Infect Dis)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# ── Build a synthetic 6-node urban network ──────────────────────────────────

N = 6
T = 200

node_types = ["core", "core", "dense", "dense", "suburban", "peripheral"]
populations = np.array([900_000, 750_000, 550_000, 450_000, 280_000, 120_000], float)

f_jk, populations, node_types = build_synthetic_network(
    N=N, node_types=node_types, populations=populations,
    T=T, seed=42, day_variation_sd=0.12,
    hub_attraction_power=0.5, decay_scale=18.0,
)
loc   = [f"L{i+1}\n({t})" for i,t in enumerate(node_types)]
loc_s = [f"L{i+1} ({t})" for i,t in enumerate(node_types)]
f0    = f_jk[0]

print(f"Network: {N} locations, {T} simulation days")
print(f"Populations (x1000): {(populations/1e3).round(0).astype(int).tolist()}")
print(f"Node types:          {node_types}")
print(f"f_jk row sums (t=0): {f_jk[0].sum(axis=1).round(6)}")

# Quick network heatmap
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
im0 = axes[0].imshow(f_jk.mean(axis=0), cmap="Blues", aspect="auto")
axes[0].set_xticks(range(N)); axes[0].set_xticklabels(loc, fontsize=7)
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(loc, fontsize=7)
axes[0].set_title("Mean mobility matrix $\\bar{f}^{jk}$")
plt.colorbar(im0, ax=axes[0]).set_label("Prob.")

axes[1].bar(range(N), populations/1e3, color=[OK[i%len(OK)] for i in range(N)])
axes[1].set_xticks(range(N)); axes[1].set_xticklabels(loc_s, fontsize=7, rotation=30, ha="right")
axes[1].set_ylabel("Population (×10³)"); axes[1].set_title("Resident populations")
plt.tight_layout(); plt.show()


## 2. β → R₀ mapping and inverting growth rate to estimate β <a id="sec2"></a>

### 2a. The β → R₀ relationship

In our framework (Mills 2026, Eqs. 12 and 14), at t=0 with a fully susceptible
population S_j(0) = N_j, the pairwise reproduction number is:

$$R^{kj}(0) = N_j \sum_{l} f^{jl}(0)\, f^{kl}(0)\, \frac{\beta\,\chi^{kl}}{N^l_\mathrm{eff}(0)}$$

where $\chi^{kl} = \lambda_W$ when the meeting location $l$ is the infector's home location $k$,
and $\chi^{kl} = \lambda_B$ otherwise, and
$N^l_\mathrm{eff}(0) = \sum_q f^{ql}(0)\,N_q$ is the effective population at meeting location $l$.

The basic reproduction number is then:

$$R_0 = \mathcal{R}(t=0) = \rho(\mathbf{R}(t=0))$$

This is what `compute_R0_from_beta` computes: it passes `lw = β·λ_W` and
`lb = β·λ_B` into the kernel (which sums over all meeting locations with the
appropriate contact rate), then takes the spectral radius of the resulting NGM.
Note that **both** λ_W and λ_B enter the kernel — the formula is not simply
proportional to β·λ_W alone.

Plotting this mapping shows how sensitive R₀ is to β for our specific network.

In [ ]:
# β → R₀ for the urban network
beta_grid = np.linspace(0.005, 0.12, 200)
R0_grid   = scan_beta_to_R0(beta_grid, f0, populations, p, LAMBDA_W, LAMBDA_B)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(beta_grid, R0_grid, color=OK[4], lw=2.0)
ax.axhline(1.0, color="#888", ls="--", lw=1.0, label="Threshold R₀=1")
for R0_ref, col, lab in [(0.8, OK[0],"0.8"), (1.2, OK[1],"1.2"),
                          (1.5, OK[2],"1.5"), (2.0, OK[5],"2.0"),
                          (3.0, OK[6],"3.0")]:
    idx = np.argmin(np.abs(R0_grid - R0_ref))
    ax.axhline(R0_ref, color=col, ls=":", lw=0.9, alpha=0.7)
    ax.scatter([beta_grid[idx]], [R0_ref], color=col, s=55, zorder=5)
    ax.annotate(f"β={beta_grid[idx]:.4f}", (beta_grid[idx], R0_ref),
                xytext=(4, 4), textcoords="offset points", fontsize=7.5, color=col)
ax.set_xlabel("β (per-contact transmission probability)", fontsize=10)
ax.set_ylabel("R₀ = ρ(R(t=0))", fontsize=10)
ax.set_title("β → R₀ mapping for the urban network", fontsize=10)
ax.legend(fontsize=8)

# Also show sensitivity dR0/dβ
ax2 = axes[1]
dR0_dbeta = np.gradient(R0_grid, beta_grid)
ax2.plot(beta_grid, dR0_dbeta, color=OK[5], lw=1.5)
ax2.set_xlabel("β", fontsize=10)
ax2.set_ylabel("dR₀/dβ  (sensitivity)", fontsize=10)
ax2.set_title("Sensitivity of R₀ to β\n(steeper = higher impact of transmission changes)", fontsize=9.5)
plt.tight_layout(); plt.show()

print("β needed for key R₀ values:")
for R0_target in [0.8, 1.0, 1.5, 2.0, 3.0]:
    idx = np.argmin(np.abs(R0_grid - R0_target))
    print(f"  R₀={R0_target:.1f}: β={beta_grid[idx]:.4f}")


### 2b. Inverting the growth rate: estimating β from observed doubling time

In the first days/weeks of an outbreak we often observe the epidemic doubling
time before R₀ is known. Using the Euler-Lotka equation and our network's
β → R₀ mapping, we can back-calculate β from the observed growth rate r.

$$1 = R_0 \sum_a p(a)\, e^{-r a} \quad\xrightarrow{\text{invert}}\quad \beta$$


In [ ]:
# Estimating β from observed doubling times
doubling_times_observed = [3, 5, 7, 10, 14, 21]  # days

print("Back-calculation of β from observed doubling times:")
print(f"{'Doubling time':>15s} | {'r (1/day)':>10s} | {'β':>10s} | {'R₀':>8s} | {'r_check':>10s}")
print("-" * 65)

beta_estimates = []
for dt in doubling_times_observed:
    r_obs = np.log(2) / dt
    try:
        est = estimate_beta_from_growth_rate(
            r_obs, p, f0, populations, LAMBDA_W, LAMBDA_B
        )
        beta_estimates.append(est)
        print(f"  {dt:>4d}d                | {r_obs:>10.4f} | {est['beta']:>10.5f} | "
              f"{est['R0']:>8.3f} | {est['r_check']:>10.4f}")
    except ValueError as e:
        print(f"  {dt:>4d}d: {e}")

# Visualise uncertainty
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

dts = [e["doubling_time"] for e in beta_estimates if not np.isnan(e["doubling_time"])]
bs  = [e["beta"] for e in beta_estimates[:len(dts)]]
R0s = [e["R0"]  for e in beta_estimates[:len(dts)]]

axes[0].plot(dts, bs, "o-", color=OK[0], lw=1.5, ms=8)
axes[0].set_xlabel("Observed doubling time (days)"); axes[0].set_ylabel("Estimated β")
axes[0].set_title("β implied by observed doubling time\n(for this network, λ_W, λ_B, and GT profile)")

axes[1].plot(dts, R0s, "s-", color=OK[4], lw=1.5, ms=8)
axes[1].axhline(1.0, color="#888", ls="--", lw=0.9)
axes[1].set_xlabel("Observed doubling time (days)"); axes[1].set_ylabel("R₀")
axes[1].set_title("R₀ implied by observed doubling time")
plt.tight_layout(); plt.show()

print("\nKey insight: the same doubling time implies different β for different networks.")
print("This is why mobility data is essential for correct inference of transmission biology.")


## 3. Full simulation in β-mode <a id="sec3"></a>

We now run the full epidemic simulation using β=0.035 as our baseline value
(corresponding to a doubling time of ~7 days for this network).
R₀ is computed as an output.


In [ ]:
# Set β from the doubling-time estimate above
BETA_BASELINE = float(estimate_beta_from_growth_rate(
    np.log(2)/7, p, f0, populations, LAMBDA_W, LAMBDA_B)["beta"])

print(f"Baseline β = {BETA_BASELINE:.5f}  (doubling time ≈ 7 days)")
R0_baseline = compute_R0_from_beta(f0, populations, p, BETA_BASELINE, LAMBDA_W, LAMBDA_B)
print(f"Implied R₀ = {R0_baseline:.4f}")

# Seed: 5 infections in the first core location
seed = np.zeros(N); seed[0] = 5.0

# Build model in β-mode
model = MobilityRtFramework(
    f_jk=f_jk, populations=populations, infectiousness_profile=p,
    contact_rate_home=LAMBDA_W, contact_rate_away=LAMBDA_B,
    initial_infections=seed, T=T,
    beta=BETA_BASELINE,   # <-- β-mode, no R0_target
    location_names=loc_s,
    verbose=True,
)
results = model.simulate()

inc  = results["incidence"]
peak = int(inc.sum(axis=1).argmax())
att  = inc.sum() / populations.sum() * 100
print(f"\nR₀ (output) = {results['R0_achieved']:.4f}")
print(f"Peak day    = {peak}")
print(f"Attack rate = {att:.1f}%")


In [ ]:
# Overview plot
fig = model.plot_overview(results)
plt.suptitle(f"β-mode simulation: β={BETA_BASELINE:.5f}, R₀={results['R0_achieved']:.3f}", 
             fontsize=11, y=1.01)
plt.show()


## 4. Univariate sensitivity analyses <a id="sec4"></a>

We now scan each parameter individually while keeping all others at baseline.
This reveals which parameters R₀, the attack rate, and the epidemic timing are
most sensitive to — critical information for early-outbreak decision-making.


In [ ]:
# ── 4a. Sweeping β ──────────────────────────────────────────────────────────

beta_sweep = np.array([0.010, 0.015, 0.020, 0.025, 0.030, 0.035, 0.040, 0.050, 0.065, 0.080])
T_sweep = 200
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

R0_betas, att_betas, peak_betas = [], [], []
Rn_trajectories = []

for beta_i in beta_sweep:
    lw_i = beta_i * LAMBDA_W; lb_i = beta_i * LAMBDA_B
    res_i = MobilityRtFramework(
        f_jk=f_jk, populations=populations, infectiousness_profile=p,
        contact_rate_home=LAMBDA_W, contact_rate_away=LAMBDA_B,
        initial_infections=seed, T=T_sweep,
        beta=beta_i, verbose=False,
    ).simulate()
    inc_i = res_i["incidence"]
    R0_betas.append(res_i["R0_achieved"])
    att_betas.append(inc_i.sum() / populations.sum() * 100)
    peak_betas.append(int(inc_i.sum(axis=1).argmax()))
    Rn_trajectories.append(res_i["R_network"])

# Panel A: R0 vs beta
axes[0].plot(beta_sweep, R0_betas, "o-", color=OK[4], lw=1.5, ms=7)
axes[0].axhline(1.0, color="#888", ls="--", lw=0.9, label="Threshold")
axes[0].set_xlabel("β"); axes[0].set_ylabel("R₀ (output)"); axes[0].set_title("R₀ vs β")
axes[0].legend()

# Panel B: attack rate vs R0
axes[1].plot(R0_betas, att_betas, "s-", color=OK[5], lw=1.5, ms=7)
axes[1].set_xlabel("R₀"); axes[1].set_ylabel("Attack rate (%)")
axes[1].set_title("Attack rate vs R₀\n(final-size relationship)")

# Panel C: R(t) trajectories
cmap = plt.cm.plasma(np.linspace(0.1, 0.9, len(beta_sweep)))
t_arr = np.arange(T_sweep)
for Rn_i, col, beta_i in zip(Rn_trajectories, cmap, beta_sweep):
    axes[2].plot(t_arr, Rn_i, color=col, lw=0.9, alpha=0.85,
                 label=f"β={beta_i:.3f}")
axes[2].axhline(1.0, color="#888", ls="--", lw=0.9)
axes[2].set_xlabel("Day $t$"); axes[2].set_ylabel("R(t)")
axes[2].set_title("R(t) trajectories under different β")
axes[2].legend(fontsize=6, ncol=2, loc="upper right")

plt.tight_layout(); plt.show()
print(f"β range tested: {beta_sweep[0]:.3f} – {beta_sweep[-1]:.3f}")
print(f"R₀ range:       {min(R0_betas):.3f} – {max(R0_betas):.3f}")
print(f"Attack rates:   {min(att_betas):.1f}% – {max(att_betas):.1f}%")


In [ ]:
# ── 4b. Sweeping λ_B/λ_W ratio ──────────────────────────────────────────────
#    β is held FIXED while the λ_B/λ_W ratio increases, so R₀ rises with the
#    ratio (more between-location transmission). The sweep therefore shows the
#    combined effect of spatial structure AND increasing overall transmission.

ratio_sweep = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.65, 0.80])
BETA_FIXED = BETA_BASELINE   # fix β, vary ratio
T_r = 200

reactivity_peaks, mixing_peaks, within_fracs = [], [], []
R0_ratios, att_ratios = [], []

for ratio_i in ratio_sweep:
    lb_i = LAMBDA_W * ratio_i
    m_i = MobilityRtFramework(
        f_jk=f_jk, populations=populations, infectiousness_profile=p,
        contact_rate_home=LAMBDA_W, contact_rate_away=lb_i,
        initial_infections=seed, T=T_r, beta=BETA_FIXED, verbose=False,
    )
    r_i = m_i.simulate()
    inc_i = r_i["incidence"]
    pk_i  = int(inc_i.sum(axis=1).argmax())
    R0_ratios.append(r_i["R0_achieved"])
    att_ratios.append(inc_i.sum() / populations.sum() * 100)
    reactivity_peaks.append(float(r_i["reactivity"][pk_i]))
    mixing_peaks.append(float(r_i["mixing_ratio"][pk_i]))
    ss_i = source_sink_analysis(r_i["R_pairwise"][pk_i])
    within_fracs.append(ss_i["pi_within"])

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, y, ylabel, title, col in [
    (axes[0], R0_ratios,        "R₀",               "R₀ vs λ_B/λ_W ratio",             OK[4]),
    (axes[1], att_ratios,       "Attack rate (%)",   "Attack rate vs λ_B/λ_W",           OK[5]),
    (axes[2], reactivity_peaks, "σ(t) at peak",      "Reactivity σ at peak\n(higher → more transient amplification)", OK[6]),
    (axes[3], within_fracs,     "π_within at peak",  "Within-location frac.\n(↑ ratio → more between-location)", OK[0]),
]:
    ax.plot(ratio_sweep, y, "o-", color=col, lw=1.4, ms=7)
    ax.set_xlabel("λ_B/λ_W ratio")
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=8.5)
plt.tight_layout(); plt.show()

print("Effect of λ_B/λ_W ratio (β fixed = {:.5f}):".format(BETA_FIXED))
for i, ratio_i in enumerate(ratio_sweep):
    print(f"  ratio={ratio_i:.2f}: R₀={R0_ratios[i]:.3f}, AR={att_ratios[i]:.1f}%, "
          f"σ_peak={reactivity_peaks[i]:.3f}, π_w={within_fracs[i]:.3f}")


In [ ]:
# ── 4c. Sweeping mean generation time (GT mean) ──────────────────────────────
#    Different pathogens / variants have different GT distributions.

gt_means   = np.array([3.0, 4.0, 5.0, 5.5, 7.0, 8.0, 10.0, 14.0])
gt_sd_fixed = 1.8
T_gt = 250

peak_days_gt, att_gt, r_gt = [], [], []

for gtm in gt_means:
    p_i = discretise_gamma(gtm, gt_sd_fixed, MAX_DAYS)
    # Recompute β for doubling time ≈ 7d with this GT profile
    try:
        est_i = estimate_beta_from_growth_rate(np.log(2)/7, p_i, f0, populations,
                                               LAMBDA_W, LAMBDA_B)
        beta_i = est_i["beta"]
    except ValueError:
        beta_i = BETA_BASELINE

    # Use f_jk[0] (static t=0 snapshot, shape (N,N)) so it broadcasts to any T_gt.
    # We are varying GT here, not mobility, so a static network is appropriate.
    m_i = MobilityRtFramework(
        f_jk=f_jk[0], populations=populations, infectiousness_profile=p_i,
        contact_rate_home=LAMBDA_W, contact_rate_away=LAMBDA_B,
        initial_infections=seed, T=T_gt, beta=beta_i, verbose=False,
    )
    r_i = m_i.simulate()
    inc_i = r_i["incidence"]
    peak_days_gt.append(int(inc_i.sum(axis=1).argmax()))
    att_gt.append(inc_i.sum() / populations.sum() * 100)
    from mobility_rt_framework import euler_lotka_r
    r_gt.append(euler_lotka_r(r_i["R_network"][3], p_i))  # early growth rate

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].plot(gt_means, peak_days_gt, "o-", color=OK[2], lw=1.4, ms=7)
axes[0].set_xlabel("GT mean (days)"); axes[0].set_ylabel("Epidemic peak day")
axes[0].set_title("Epidemic peak day vs GT mean\n(β re-calibrated to 7d doubling)")

axes[1].plot(gt_means, att_gt, "s-", color=OK[5], lw=1.4, ms=7)
axes[1].set_xlabel("GT mean (days)"); axes[1].set_ylabel("Attack rate (%)")
axes[1].set_title("Attack rate vs GT mean")

axes[2].plot(gt_means, [x for x in r_gt], "^-", color=OK[0], lw=1.4, ms=7)
axes[2].set_xlabel("GT mean (days)"); axes[2].set_ylabel("Growth rate r(t) [early]")
axes[2].set_title("Early growth rate vs GT mean\n(held ≈ ln2/7 by β-recalibration → flat)")
plt.tight_layout(); plt.show()

print("Key insight: longer generation times → later, flatter epidemic peak")
print("(β is re-calibrated per GT to fix the 7d doubling time, so r is held\n"
      " ≈ ln2/7 ≈ 0.099/day and longer GT requires a higher β / R₀)")

## 5. Bivariate parameter surfaces <a id="sec5"></a>

We now run 2D parameter sweeps to visualise how outputs vary across *pairs*
of uncertain parameters. These surface plots are the core of the parameter
exploration tool — they help identify:
- **Iso-R₀ contours**: which (β, ratio) combinations give R₀ = 1?
- **Attack rate ridges**: parameter combinations leading to large outbreaks
- **Reactivity hotspots**: where transient amplification risk is highest


In [ ]:
# ── 5a. β × λ_B/λ_W → R₀ and attack rate surfaces ──────────────────────────

N_grid = 12   # grid resolution per axis
beta_ax  = np.linspace(0.008, 0.09, N_grid)
ratio_ax = np.linspace(0.05, 0.75, N_grid)

R0_surf   = np.zeros((N_grid, N_grid))
att_surf  = np.zeros((N_grid, N_grid))
peak_surf = np.zeros((N_grid, N_grid))
react_surf = np.zeros((N_grid, N_grid))

T_surf = 180
for i, beta_i in enumerate(beta_ax):
    for j, ratio_i in enumerate(ratio_ax):
        lb_i = LAMBDA_W * ratio_i
        R0_surf[i,j] = compute_R0_from_beta(f0, populations, p, beta_i, LAMBDA_W, lb_i)
        try:
            m_ij = MobilityRtFramework(
                f_jk=f_jk, populations=populations, infectiousness_profile=p,
                contact_rate_home=LAMBDA_W, contact_rate_away=lb_i,
                initial_infections=seed, T=T_surf, beta=beta_i, verbose=False,
            )
            r_ij = m_ij.simulate()
            inc_ij = r_ij["incidence"]
            att_surf[i,j]   = inc_ij.sum() / populations.sum() * 100
            pk_ij           = int(inc_ij.sum(axis=1).argmax())
            peak_surf[i,j]  = pk_ij
            react_surf[i,j] = float(r_ij["reactivity"][pk_ij])
        except Exception:
            att_surf[i,j]   = np.nan
            peak_surf[i,j]  = np.nan
            react_surf[i,j] = np.nan

print("Parameter sweep complete.")
print(f"R₀ range: [{np.nanmin(R0_surf):.2f}, {np.nanmax(R0_surf):.2f}]")
print(f"Attack rate range: [{np.nanmin(att_surf):.1f}%, {np.nanmax(att_surf):.1f}%]")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

XX, YY = np.meshgrid(ratio_ax, beta_ax)
labels = [r"R₀ = ρ(R(0))", "Attack rate (%)", "Peak day", "Reactivity σ at peak"]
Zs     = [R0_surf, att_surf, peak_surf, react_surf]
cmaps  = ["RdBu_r", "YlOrRd", "viridis_r", "plasma"]

for ax, Z, lab, cmap in zip(axes, Zs, labels, cmaps):
    im = ax.pcolormesh(ratio_ax, beta_ax, Z, cmap=cmap, shading="auto")
    # R₀=1 contour on all panels
    if lab == "R₀ = ρ(R(0))":
        cs = ax.contour(ratio_ax, beta_ax, Z, levels=[1.0], colors=["white"],
                        linewidths=2.0, linestyles="--")
        ax.clabel(cs, fmt="R₀=1", fontsize=8.5, colors="white")
    else:
        # Overlay R₀ contours
        cs = ax.contour(ratio_ax, beta_ax, R0_surf,
                        levels=[0.8, 1.0, 1.5, 2.0, 2.5, 3.0],
                        colors="white", linewidths=0.8, alpha=0.7)
        ax.clabel(cs, fmt="R₀=%.1f", fontsize=7, colors="white")
    ax.set_xlabel("λ_B/λ_W ratio"); ax.set_ylabel("β")
    ax.set_title(lab, fontsize=9.5)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Parameter surface: β × λ_B/λ_W ratio\n(white contours = iso-R₀ lines)",
             fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

print("Interpretation:")
print("  Upper-left region (high β, low ratio): high R₀, mostly local transmission")
print("  Upper-right (high β, high ratio): high R₀, more spatial spread, higher reactivity")
print("  Lower region (low β): sub-critical, no major outbreak")


In [ ]:
# ── 5b. GT mean × GT SD → epidemic speed surface ────────────────────────────
#    Fix β to give R₀≈1.5 with baseline GT, then vary GT shape.

gt_mean_ax = np.array([3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0])
gt_sd_ax   = np.array([0.8, 1.2, 1.8, 2.4, 3.0])
N_gm, N_gs = len(gt_mean_ax), len(gt_sd_ax)

peak_gt_surf   = np.zeros((N_gm, N_gs))
r_gt_surf      = np.zeros((N_gm, N_gs))
T_gt2 = 200

for i, gm in enumerate(gt_mean_ax):
    for j, gs in enumerate(gt_sd_ax):
        p_ij = discretise_gamma(gm, gs, MAX_DAYS)
        # Back-calculate β for doubling time 7d with this GT
        try:
            est_ij = estimate_beta_from_growth_rate(np.log(2)/7, p_ij, f0,
                                                     populations, LAMBDA_W, LAMBDA_B)
            beta_ij = est_ij["beta"]
        except ValueError:
            beta_ij = BETA_BASELINE

        m_ij = MobilityRtFramework(
            f_jk=f_jk, populations=populations, infectiousness_profile=p_ij,
            contact_rate_home=LAMBDA_W, contact_rate_away=LAMBDA_B,
            initial_infections=seed, T=T_gt2, beta=beta_ij, verbose=False,
        )
        r_ij = m_ij.simulate()
        inc_ij = r_ij["incidence"]
        peak_gt_surf[i,j] = int(inc_ij.sum(axis=1).argmax())
        from mobility_rt_framework import euler_lotka_r
        r_gt_surf[i,j] = euler_lotka_r(r_ij["R_network"][2], p_ij)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
XX2, YY2 = np.meshgrid(gt_sd_ax, gt_mean_ax)
for ax, Z, title, cmap in [
    (axes[0], peak_gt_surf, "Epidemic peak day", "viridis_r"),
    (axes[1], r_gt_surf,   "Early growth rate r(t) [1/day]\n(≈ ln2/7, fixed by β-calibration)", "YlOrRd"),
]:
    im = ax.pcolormesh(gt_sd_ax, gt_mean_ax, Z, cmap=cmap, shading="auto")
    ax.set_xlabel("GT standard deviation (days)"); ax.set_ylabel("GT mean (days)")
    ax.set_title(title, fontsize=9.5)
    plt.colorbar(im, ax=ax)
plt.suptitle("Generation time surface (β calibrated to doubling time = 7d)", fontsize=10)
plt.tight_layout(); plt.show()

print("\nLonger mean GT → later peak; the early growth rate r is held ≈ ln2/7 by")
print("β-recalibration (so longer GT / higher GT SD instead require a higher β / R₀).")


## 6. Cross-network comparison <a id="sec6"></a>

Here we compare the dense urban network with a sparse rural network under the
**same β** (same transmission biology). The differences in epidemic dynamics are
purely structural — driven by the different mobility and population patterns.

This illustrates why it is wrong to compare R₀ values across settings without
accounting for mobility structure: the same β produces very different R₀ and
epidemic trajectories in different network contexts.


In [ ]:
# Build the sparse rural network
node_types_B = ["capital","peri-capital","rural","rural","semi-rural","remote"]
pop_B  = np.array([3_000_000, 1_500_000, 800_000, 600_000, 350_000, 200_000], float)
cfrac_B = np.array([0.06, 0.07, 0.025, 0.02, 0.015, 0.01])

f_jk_B, pop_B, types_B = build_synthetic_network(
    N=N, node_types=node_types_B, populations=pop_B, T=T,
    commuting_fracs=cfrac_B, decay_scale=150.0, seed=99, day_variation_sd=0.08,
)
loc_B  = [f"L{i+1} ({t})" for i,t in enumerate(types_B)]
f0_B   = f_jk_B[0]
seed_B = np.zeros(N); seed_B[0] = 5.0

# Compute R₀ for SAME β on BOTH networks
R0_urban_beta  = compute_R0_from_beta(f0,   populations, p, BETA_BASELINE, LAMBDA_W, LAMBDA_B)
R0_rural_beta  = compute_R0_from_beta(f0_B, pop_B,       p, BETA_BASELINE, LAMBDA_W, LAMBDA_B)
print(f"Same β={BETA_BASELINE:.5f}, same biology (POLYMOD + Hart 2022 GT):")
print(f"  R₀ urban network: {R0_urban_beta:.4f}")
print(f"  R₀ rural network: {R0_rural_beta:.4f}")
print(f"  Difference: {abs(R0_urban_beta - R0_rural_beta):.4f}")
print("→ R₀ differs due to different mobility structure, not different biology!")

model_B = MobilityRtFramework(
    f_jk=f_jk_B, populations=pop_B, infectiousness_profile=p,
    contact_rate_home=LAMBDA_W, contact_rate_away=LAMBDA_B,
    initial_infections=seed_B, T=T,
    beta=BETA_BASELINE, location_names=loc_B, verbose=False,
)
results_B = model_B.simulate()

# Compare key outputs
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
t_arr = np.arange(T)

for row, (res, label, pops) in enumerate([
    (results,   "Dense urban", populations),
    (results_B, "Sparse rural", pop_B),
]):
    inc_x = res["incidence"]
    Rn_x  = res["R_network"]
    sg_x  = res["reactivity"]
    Et_x  = res["E_risk_averse"]

    # R(t), sigma(t), E(t)
    axes[row,0].plot(t_arr, Rn_x, color=OK[4], lw=1.3, label=r"$\mathcal{R}(t)$")
    axes[row,0].plot(t_arr, sg_x, color=OK[5], lw=1.0, ls="--", label=r"$\sigma(t)$")
    axes[row,0].plot(t_arr, Et_x, color=OK[6], lw=0.9, ls=":",  label=r"$\mathcal{E}(t)$")
    axes[row,0].axhline(1.0, color="#777", ls="--", lw=0.8)
    axes[row,0].set_title(f"{label}: R(t), σ(t), E(t)", fontsize=9)
    axes[row,0].legend(fontsize=7, ncol=2)
    axes[row,0].set_xlabel("Day $t$")

    # Incidence heatmap
    N_x = inc_x.shape[1]
    loc_x = [f"L{j+1}" for j in range(N_x)]
    axes[row,1].imshow(inc_x.T, aspect="auto", cmap="YlOrRd", origin="upper",
                       extent=[-0.5,T-0.5,N_x-0.5,-0.5])
    axes[row,1].set_yticks(range(N_x)); axes[row,1].set_yticklabels(loc_x, fontsize=8)
    axes[row,1].set_xlabel("Day $t$"); axes[row,1].set_ylabel("Location $j$")
    axes[row,1].set_title(f"{label}: incidence by location", fontsize=9)

    # R_out vs R_in at peak
    pk_x = int(inc_x.sum(axis=1).argmax())
    axes[row,2].scatter(res["R_outward"][pk_x], res["R_inward"][pk_x],
                        s=90, c=np.arange(N_x), cmap="tab10",
                        edgecolors="#222", linewidths=0.6)
    for j in range(N_x):
        axes[row,2].annotate(f"L{j+1}", (res["R_outward"][pk_x,j], res["R_inward"][pk_x,j]),
                             textcoords="offset points", xytext=(4,3), fontsize=7.5)
    m_xy = max(res["R_outward"][pk_x].max(), res["R_inward"][pk_x].max()) * 1.05
    axes[row,2].plot([0,m_xy],[0,m_xy], color="#aaa", ls="--", lw=0.8)
    axes[row,2].set_xlabel(r"$R^k_{\rm out}$"); axes[row,2].set_ylabel(r"$R^j_{\rm in}$")
    axes[row,2].set_title(f"{label}: source-sink at peak (day {pk_x})", fontsize=9)

plt.suptitle(f"Cross-network comparison (SAME β={BETA_BASELINE:.5f})", fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

print("\nComparison at epidemic peak:")
pk_u = int(results["incidence"].sum(axis=1).argmax())
pk_r = int(results_B["incidence"].sum(axis=1).argmax())
print(f"  Urban: peak day {pk_u}, AR={results['incidence'].sum()/populations.sum()*100:.1f}%, "
      f"R₀={R0_urban_beta:.3f}")
print(f"  Rural: peak day {pk_r}, AR={results_B['incidence'].sum()/pop_B.sum()*100:.1f}%, "
      f"R₀={R0_rural_beta:.3f}")


## 7. Uncertainty propagation: β uncertainty → R₀ uncertainty <a id="sec7"></a>

Early in an outbreak the observed growth rate carries uncertainty.
We propagate this uncertainty through β → R₀ to obtain a credible interval
for R₀ based on the uncertainty in the doubling time observation.

This is a **practical framework for outbreak start**: given a doubling time
estimate with associated uncertainty, what is the range of plausible R₀ values?


In [ ]:
# Simulate uncertainty: doubling time = 7 ± 2 days (normal, truncated positive)
DT_MEAN = 7.0   # days
DT_SD   = 2.0   # days — uncertainty in early growth observation
N_SAMPLES = 500

rng = np.random.default_rng(2026)
dt_samples = rng.normal(DT_MEAN, DT_SD, N_SAMPLES)
dt_samples = dt_samples[dt_samples > 1.5]  # truncate implausibly fast

beta_samples, R0_samples = [], []
for dt_s in dt_samples:
    r_s = np.log(2) / dt_s
    try:
        est_s = estimate_beta_from_growth_rate(r_s, p, f0, populations, LAMBDA_W, LAMBDA_B)
        beta_samples.append(est_s["beta"])
        R0_samples.append(est_s["R0"])
    except ValueError:
        pass

beta_arr = np.array(beta_samples)
R0_arr   = np.array(R0_samples)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Histogram of β
axes[0].hist(beta_arr, bins=40, color=OK[0], edgecolor="none", alpha=0.8, density=True)
axes[0].axvline(np.median(beta_arr), color=OK[5], lw=1.5, label=f"Median β={np.median(beta_arr):.4f}")
axes[0].axvline(np.percentile(beta_arr, 5),  color=OK[5], lw=0.9, ls=":")
axes[0].axvline(np.percentile(beta_arr, 95), color=OK[5], lw=0.9, ls=":")
axes[0].set_xlabel("β"); axes[0].set_ylabel("Density")
axes[0].set_title(f"Implied β distribution\n(doubling time = {DT_MEAN}±{DT_SD}d)")
axes[0].legend(fontsize=8)

# Histogram of R₀
axes[1].hist(R0_arr, bins=40, color=OK[4], edgecolor="none", alpha=0.8, density=True)
axes[1].axvline(np.median(R0_arr), color="k", lw=1.5, label=f"Median R₀={np.median(R0_arr):.3f}")
axes[1].axvline(np.percentile(R0_arr, 5),  color="k", lw=0.9, ls=":")
axes[1].axvline(np.percentile(R0_arr, 95), color="k", lw=0.9, ls=":")
axes[1].axvline(1.0, color=OK[5], lw=0.9, ls="--", label="R₀=1 threshold")
axes[1].set_xlabel("R₀"); axes[1].set_ylabel("Density")
axes[1].set_title(f"Implied R₀ distribution\n90% CI: [{np.percentile(R0_arr,5):.2f}, {np.percentile(R0_arr,95):.2f}]")
axes[1].legend(fontsize=8)

# Joint β-R₀ relationship with uncertainty band
axes[2].scatter(beta_arr, R0_arr, c=dt_samples[:len(beta_arr)], cmap="coolwarm",
                s=8, alpha=0.4)
axes[2].plot(beta_grid, R0_grid, color="k", lw=1.5, label="True R₀(β) for urban network")
axes[2].axhline(1.0, color="#888", ls="--", lw=0.9)
axes[2].set_xlabel("β"); axes[2].set_ylabel("R₀")
axes[2].set_title("β–R₀ scatter from doubling-time uncertainty\n(colour = doubling time observed)")
axes[2].legend(fontsize=8)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(dt_samples[:len(beta_arr)].min(), dt_samples[:len(beta_arr)].max()))
sm.set_array([]); plt.colorbar(sm, ax=axes[2], label="Doubling time (days)")

plt.tight_layout(); plt.show()

print(f"Summary from {len(R0_arr)} samples (doubling time = {DT_MEAN}±{DT_SD}d):")
print(f"  β:  [{np.percentile(beta_arr,5):.5f}, {np.percentile(beta_arr,95):.5f}] (90% CI)")
print(f"  R₀: [{np.percentile(R0_arr,5):.3f}, {np.percentile(R0_arr,95):.3f}] (90% CI)")
print(f"  Pr(R₀>1): {(R0_arr > 1).mean()*100:.1f}%")


## Summary

This notebook demonstrated the **β-first parametrisation** approach:

### Key framework extensions
| Feature | Description |
|---------|-------------|
| `beta=` parameter | Pass per-contact transmission probability directly |
| `compute_R0_from_beta()` | Compute R₀ from β without running a full simulation |
| `estimate_beta_from_growth_rate()` | Invert observed doubling time → β via Euler-Lotka |
| `scan_beta_to_R0()` | Fast vectorised β → R₀ mapping |
| `parameter_sweep()` | Full-factorial sweep across any parameter axes |

### What's next: the dashboard
The next natural step is an **interactive dashboard** where users can explore
all these parameter axes in real time using sliders. See `dashboard_plan.md`
for the full specification.

```python
# Quick dashboard preview (static — see dashboard_plan.md for full Streamlit spec)
from mobility_rt_framework import parameter_sweep
sweep = parameter_sweep(
    f_jk_series = f_jk,
    populations  = populations,
    initial_infections = seed,
    T            = 200,
    param_grid   = {
        'beta':       np.linspace(0.01, 0.08, 8),
        'lb_lw_ratio': np.linspace(0.1,  0.6,  6),
    },
    contact_rate_home  = 13.0,
    contact_rate_away_ratio = 0.30,
    verbose = True,
)
```
